In [2]:
import pandas as pd

# Load the CSV from the project's data folder.
df = pd.read_csv("../data/writing_pairs.csv")

# Remove rows where both versions are missing.
df = df.dropna(subset=["AI Version", "Ethan Rewrite"], how="all")

# Count words in each version. Missing text counts as zero words.
df["AI Words"] = df["AI Version"].fillna("").str.split().str.len()
df["Rewrite Words"] = df["Ethan Rewrite"].fillna("").str.split().str.len()

print("Total pairs:", len(df))

# Show the sample IDs, text types, and word counts.
df[["ID", "Type", "AI Words", "Rewrite Words"]]

Total pairs: 10


,ID,Type,AI Words,Rewrite Words
0,1,LinkedIn Post,219,259
1,2,LinkedIn Post,235,264
2,3,LinkedIn Post,228,281
3,4,LinkedIn Post,234,306
4,5,LinkedIn Post,230,288
5,6,LinkedIn Post,221,272
6,7,LinkedIn Post,223,255
7,8,LinkedIn Post,223,305
8,9,LinkedIn Post,219,283
9,10,LinkedIn Post,192,115


In [3]:
# Check both text columns for missing or blank entries.
for column in ["AI Version", "Ethan Rewrite"]:
    blank = df[column].fillna("").str.strip().eq("")
    print(f"Blank {column} entries: {blank.sum()}")

# Check whether any sample IDs are repeated.
print("Duplicate IDs:", df["ID"].duplicated().sum())

# Check whether any rows contain the same input and rewrite.
print(
    "Duplicate pairs:",
    df.duplicated(subset=["AI Version", "Ethan Rewrite"]).sum()
)

Blank AI Version entries: 0
Blank Ethan Rewrite entries: 0
Duplicate IDs: 0
Duplicate pairs: 0


In [4]:
import sys

# Show which Python version this notebook is using.
print("Python version:", sys.version)

try:
    import torch

    # Check whether PyTorch is installed and can access a GPU.
    print("PyTorch version:", torch.__version__)
    print("GPU available:", torch.cuda.is_available())

    # Print the GPU name if one is available.
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

except ImportError:
    print("PyTorch is not installed in this notebook's environment.")

Python version: 3.11.5 (main, Apr  5 2024, 11:25:47) [GCC 13.2.0]
PyTorch version: 2.7.0+cu126
GPU available: True
GPU: NVIDIA A30


In [5]:
from importlib.metadata import version, PackageNotFoundError

# Show the installed versions of the packages used to load the model.
for package in ["transformers", "accelerate", "huggingface-hub"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

transformers: 4.56.1
accelerate: 1.10.0
huggingface-hub: 0.36.2


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Choose the model used for the rewrite experiments.
model_id = "Qwen/Qwen2.5-3B-Instruct"

# The tokenizer converts text into tokens the model can process.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model, downloading its files if they are not already cached.
# bfloat16 reduces memory use compared with 32-bit precision.
# device_map="auto" lets the library place the model on available hardware.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

# Use evaluation mode because this cell is not training the model.
model.eval()

print("Model loaded!")
print("Device:", model.device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!
Device: cuda:0


In [7]:
# Use the first sample's AI text as the input.
original = df.iloc[0]["AI Version"]

# Give general rewrite instructions without personal writing examples.
messages = [
    {
        "role": "system",
        "content": (
            "Rewrite LinkedIn posts in clear, natural language. "
            "Use everyday words and varied sentence lengths. "
            "Keep the original meaning and facts. "
            "Do not invent experiences or add claims. "
            "Return only the rewritten post."
        )
    },
    {"role": "user", "content": original}
]

# Format the messages for this model and convert them into tokens.
# Move the inputs to the same device as the model.
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# Generate text without tracking gradients, since we are not training.
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=600,  # Limit the output to 600 new tokens, not words.
        do_sample=False,    # Choose the highest-scoring next token.
        pad_token_id=tokenizer.eos_token_id
    )

# Remove the input tokens so only the new response remains.
new_tokens = output[0, inputs["input_ids"].shape[1]:]

# Convert the response tokens back into readable text.
baseline_rewrite = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print(baseline_rewrite)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Recently, I got to work on a project that really helped me understand how technology can make our daily tasks easier and more efficient. We looked at a task that usually requires a lot of manual effort and figured out ways to automate it so it’s quicker and more reliable.

During the project, I got to practice using automation tools, doing tests, and figuring out how everything should flow from beginning to end. One thing I realized is that just getting the automation to work isn’t enough. It needs to be simple for others to use, understand, and keep up with too.

A big part of the project involved fixing little glitches, testing various scenarios, and making adjustments based on feedback. Some parts took longer than I thought they would, but those challenges were actually very helpful. They taught me to take my time, figure out exactly what’s wrong, and come up with a better solution.

Overall, this project improved my tech skills and how I tackle problems. I’m excited to keep learnin

In [8]:
from pathlib import Path
import json

# Create the results folder if it does not exist.
results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

# Keep the input, output, reference, and settings together.
# The reference rewrite is saved for comparison, not sent to the model.
result = {
    "sample_id": str(df.iloc[0]["ID"]),
    "model": model_id,
    "method": "basic_prompt",
    "messages": messages,
    "max_new_tokens": 600,
    "do_sample": False,
    "original_ai": original,
    "reference_rewrite": df.iloc[0]["Ethan Rewrite"],
    "model_rewrite": baseline_rewrite
}

# Save as readable JSON. Rerunning replaces the file at this path.
with open(
    results_dir / "baseline_sample_001.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("Saved baseline result!")

Saved baseline result!


In [9]:
# Keep the baseline instructions and add instructions to follow examples.
style_messages = [
    {
        "role": "system",
        "content": (
            messages[0]["content"]
            + " Follow the writing style shown in the example rewrites. "
            "Use their style, but do not copy their facts into the new post."
        )
    }
]

# Use rows 2–4 as example input/rewrite pairs.
# Row 1's reference rewrite is left out so the model cannot copy it.
for _, row in df.iloc[1:4].iterrows():
    style_messages.extend([
        {"role": "user", "content": row["AI Version"]},
        {"role": "assistant", "content": row["Ethan Rewrite"]}
    ])

# Ask the model to rewrite the same input used for the baseline.
style_messages.append({"role": "user", "content": original})

# Convert the instructions, examples, and input into model tokens.
style_inputs = tokenizer.apply_chat_template(
    style_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# Use the same generation settings as the baseline.
# The examples are part of the prompt; model weights are not changed.
with torch.inference_mode():
    style_output = model.generate(
        **style_inputs,
        max_new_tokens=600,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Keep only the generated response and turn it into text.
style_tokens = style_output[0, style_inputs["input_ids"].shape[1]:]
example_rewrite = tokenizer.decode(
    style_tokens,
    skip_special_tokens=True
)

print(example_rewrite)

Recently, I had the chance to work on an automation project that really helped me understand how technology can make everyday tasks easier and more efficient.

The project involved identifying a process that usually requires a lot of manual work and figuring out a way to automate it. Along the way, I got to practice using automation tools, testing, debugging, and planning out the entire process from beginning to end.

One of the key lessons I learned was that making automation work isn’t just about coding—it’s also about making sure it’s user-friendly and easy to maintain. I spent a lot of time tweaking the system, testing various scenarios, and adjusting based on feedback from others.

Some parts of the project took longer than I anticipated, but that was actually one of the most valuable parts of the experience. Each issue forced me to take a step back, figure out the root cause, and come up with a better solution.

Overall, this project helped me grow both technically and in how I t

In [10]:
# Record the examples and settings used so this run can be reviewed later.
example_result = {
    "sample_id": str(df.iloc[0]["ID"]),
    "model": model_id,
    "method": "example_prompt",
    "example_ids": df.iloc[1:4]["ID"].astype(str).tolist(),
    "messages": style_messages,
    "max_new_tokens": 600,
    "do_sample": False,
    "original_ai": original,
    "reference_rewrite": df.iloc[0]["Ethan Rewrite"],
    "model_rewrite": example_rewrite
}

# Save separately from the baseline. Rerunning replaces this file.
with open(
    results_dir / "example_prompt_sample_001.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(example_result, f, ensure_ascii=False, indent=2)

print("Saved example-based result!")

Saved example-based result!


In [11]:
# Display the reference and both model outputs for manual comparison.
# Look at wording, sentence flow, tone, and whether details changed.
print("YOUR REWRITE:\n")
print(df.iloc[0]["Ethan Rewrite"])

print("\nBASELINE:\n")
print(baseline_rewrite)

print("\nWITH EXAMPLES:\n")
print(example_rewrite)

YOUR REWRITE:

Recently I had the chance to go and work on this automation project that allowed me to learn much more about how technology is able to be used to make everyday processes much better

The project I was tasked with was finding a certain process that needs a lot of manual work and I designed and developed a way to make it both faster and more efficient. While doing this, I learned how automation tools work and the skills that go along with it such as testing, debugging, problem solving, etc.

The biggest thing I learned throughout making this project was that making the automation work is one of the easiest steps in the development process. These automations have to be easy for other people to use, maintain, or update as well, especially people that aren't so familiar with technology. There was quite a bit of time that went into testing the code or fixing various smaller bugs.

Of course, there were definitely a few parts of the project that took much more time than I had e

In [12]:
from pathlib import Path

# This starter split uses the current row order.
# Keep related or near-duplicate posts in the same group.
# The small groups are for testing the workflow, not strong final results.

# First six pairs: examples for prompting, RAG, and later training.
train_df = df.iloc[:6].copy()

# Next two pairs: check results while adjusting the system.
validation_df = df.iloc[6:8].copy()

# Last two pairs: reserve for the final comparison.
# Do not use these as training data or prompt examples.
test_df = df.iloc[8:].copy()

# Create a folder for the separate dataset files.
split_dir = Path("../data/splits")
split_dir.mkdir(parents=True, exist_ok=True)

# Save the original columns without the calculated word counts.
columns = ["ID", "Type", "AI Version", "Ethan Rewrite"]

for name, subset in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df)
]:
    subset[columns].to_csv(split_dir / f"{name}.csv", index=False)
    print(f"{name}: {len(subset)} pairs")

train: 6 pairs
validation: 2 pairs
test: 2 pairs
